# Домашка
1. Пофиксить loss nan (поиграться с фичами в CustomDataset)
2. Добавить в `fit` проверку на valid

In [ ]:
!pip install torchmetrics tensorboard --quiet
%load_ext tensorboard
!rm -rf runs

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader

from torchmetrics import Accuracy

## CustomDataset

In [ ]:
class CustomDataset(Dataset):
    def __init__(
        self,
        path: str,
        num_cols: list[str],
        log1p_cols: list[str],
        n_categories_of: dict[str, int],
    ) -> None:

        data = pd.read_parquet(path)

        cat_cols = list(n_categories_of.keys())

        X = data[num_cols + cat_cols].reset_index(drop=True)

        for col in num_cols:
            if X[col].isna().any():
                X[col] = X[col].fillna(X[col].median())

        for col in log1p_cols:
            X[col] = np.log1p(X[col])

        self.X = X.to_numpy(dtype=np.float32)
        self.y = np.asarray(
            [0 if el == "<50k" else 1 for el in data["salary"].values],
            dtype=np.float32,
        )

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        features = torch.from_numpy(self.X[idx])
        label = torch.from_numpy(self.y[idx:idx+1])
        return features, label

## Архитектура

In [ ]:
class DenseFeatureLayer(nn.Module):
    def __init__(
        self,
        num_cols: list[str],
        n_categories_of: dict[str, int],
        emb_size: int,
    ) -> None:
        super().__init__()

        self.num_cols = num_cols
        self.cat_cols = list(n_categories_of.keys())

        self.numeric_feature_bn = nn.BatchNorm1d(len(num_cols), eps=1e-3)

        n_features = len(self.num_cols) + len(self.cat_cols)
        self.first_feature_bn = nn.BatchNorm1d(n_features, eps=1e-3)
        self.second_feature_bn = nn.BatchNorm1d(n_features, eps=1e-3)

        # first order feature interactions
        self.first_order_embd = nn.ModuleDict()
        for col in self.cat_cols:
            self.first_order_embd[col] = nn.Embedding(n_categories_of[col], 1)
        self.first_order_scalar = nn.ParameterDict({})
        for col in num_cols:
            self.first_order_scalar[col] = nn.Parameter(nn.init.xavier_uniform_(torch.empty(1, 1)))

        # second order feature interactions
        self.second_order_embd = nn.ModuleDict({})
        for col in self.cat_cols:
            self.second_order_embd[col] = nn.Embedding(n_categories_of[col], emb_size)
        self.second_order_scalar = nn.ParameterDict({})
        for col in num_cols:
            self.second_order_scalar[col] = nn.Parameter(nn.init.xavier_uniform_(torch.empty(emb_size, 1)))

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:

        num_feats, cat_feats = torch.split(x, [len(self.num_cols), len(self.cat_cols)], dim=1)
        cat_feats = cat_feats.long()

        num_feats = self.numeric_feature_bn(num_feats)

        # first order feature interactions
        first_order_parts = []
        # categorical_columns
        for i, col in enumerate(self.cat_cols):
            first_order_parts.append(self.first_order_embd[col](cat_feats[:, i]))
        # numeric_columns
        for i, col in enumerate(self.num_cols):
            first_order_parts.append(
                (num_feats[:, i] * self.first_order_scalar[col].squeeze(-1)).unsqueeze(1)
            )
        first_order_embd_output = torch.cat(first_order_parts, dim=1)

        # second order feature interactions
        second_order_parts = []
        # categorical_columns
        for i, col in enumerate(self.cat_cols):
            second_order_parts.append(
                self.second_order_embd[col](cat_feats[:, i]).unsqueeze(1)
            )
        # numeric_columns
        for i, col in enumerate(self.num_cols):
            scalar = self.second_order_scalar[col].squeeze(-1)
            col_feature = num_feats[:, i].unsqueeze(1).unsqueeze(2)
            second_order_parts.append(col_feature * scalar.unsqueeze(0).unsqueeze(0))
        second_order_embd_output = torch.cat(second_order_parts, dim=1)

        first_order_embd_output = self.first_feature_bn(first_order_embd_output)
        second_order_embd_output = self.second_feature_bn(second_order_embd_output)

        return first_order_embd_output, second_order_embd_output

In [ ]:
class FMLayer(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, first_order_emb: Tensor, second_order_emb: Tensor) -> tuple[Tensor, Tensor]:

        first_order = first_order_emb.sum(dim=1, keepdim=True)

        sum_sq = second_order_emb.sum(dim=1) ** 2
        sq_sum = (second_order_emb ** 2).sum(dim=1)
        second_order = 0.5 * (sum_sq - sq_sum)

        return first_order, second_order

In [ ]:
class MLPLayer(nn.Module):
    def __init__(
        self,
        input_size: int,
        n_layers: int,
        n_neurons: int,
        output_size: int,
    ) -> None:
        super().__init__()

        sizes = [input_size] + [n_neurons] * (n_layers - 1)
        layers = []
        for in_size, out_size in zip(sizes, sizes[1:]):
            linear = nn.Linear(in_size, out_size)
            nn.init.xavier_uniform_(linear.weight, gain=nn.init.calculate_gain("relu"))
            nn.init.zeros_(linear.bias)
            layers.extend([linear, nn.BatchNorm1d(out_size), nn.ReLU()])

        self.deep_block = nn.Sequential(*layers)
        self.output_layer = nn.Linear(n_neurons, output_size)
        nn.init.xavier_uniform_(self.output_layer.weight)
        nn.init.zeros_(self.output_layer.bias)

    def forward(self, x):
        h = self.deep_block(x)
        h = self.output_layer(h)
        return h

In [ ]:
class DeepFMNet(nn.Module):
    def __init__(
        self,
        num_cols: list[str],
        n_categories_of: dict[str, int],
        emb_size: int,
        n_layers: int,
        n_neurons: int,
        output_size: int,
        n_classes: int,
    ) -> None:
        super().__init__()

        self.num_cols = num_cols
        self.cat_cols = list(n_categories_of.keys())
        self.emb_size = emb_size

        self.features_emb = DenseFeatureLayer(num_cols, n_categories_of, emb_size)
        self.FM = FMLayer()

        input_size = (len(self.cat_cols) + len(self.num_cols)) * emb_size
        self.MLP = MLPLayer(input_size, n_layers, n_neurons, output_size)

        input_size = 1 + emb_size + output_size
        self.dense_layer = nn.Linear(input_size, n_classes)

    def forward(self, x: Tensor) -> Tensor:
        first_order_emb, second_order_emb = self.features_emb(x)

        FM_first_order, FM_second_order = self.FM(first_order_emb, second_order_emb)

        second_order_emb = second_order_emb.flatten(1)

        Deep = self.MLP(second_order_emb)

        concat_out = torch.cat([FM_first_order, FM_second_order, Deep], dim=1)

        out = self.dense_layer(concat_out)

        return out

## Train Loop

In [ ]:
TRAIN_DATA_PATH = "train_adult.parquet"
VALID_DATA_PATH = "valid_adult.parquet"
N_CATEGORIES_OF = {
    "workclass_cat": 9,
    "education_cat": 16,
    "marital-status_cat": 7,
    "occupation_cat": 16,
    "relationship_cat": 6,
    "race_cat": 5,
    "sex_cat": 2,
    "native-country_cat": 42,
}
NUM_COLS = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]
LOG1P_COLS = ["fnlwgt", "capital-gain", "capital-loss"]

BATCH_SIZE = 512

In [ ]:
train_dataset = CustomDataset(TRAIN_DATA_PATH, NUM_COLS, LOG1P_COLS, N_CATEGORIES_OF)
train_dataloader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)

valid_dataset = CustomDataset(VALID_DATA_PATH, NUM_COLS, LOG1P_COLS, N_CATEGORIES_OF)
valid_dataloader = DataLoader(valid_dataset, BATCH_SIZE)

In [ ]:
EPOCHS = 50
EMB_SIZE = 5
N_LAYERS = 3
N_NEURONS = 50
DEEP_OUTPUT_SIZE = 50
N_CLASSES = 1
LEARNING_RATE = 3e-4

In [ ]:
class DeepFM:
    def __init__(
        self,
        num_cols: list[str],
        n_categories_of: dict[str, int],
        emb_size: int,
        n_layers: int,
        n_neurons: int,
        deep_output_size: int,
        n_classes: int,
        learning_rate: float,
    ) -> None:
        self.net = DeepFMNet(num_cols, n_categories_of, emb_size, n_layers, n_neurons, deep_output_size, n_classes)
        self.loss = nn.BCEWithLogitsLoss()
        self.train_accuracy = Accuracy(task="binary")
        self.valid_accuracy = Accuracy(task="binary")
        self.optimizer = optim.Adam(self.net.parameters(), lr=learning_rate)
        self.train_writer = SummaryWriter("runs/train")
        self.valid_writer = SummaryWriter("runs/valid")

    def fit(self, train_loader: DataLoader, valid_loader: DataLoader, epochs: int) -> None:
        step = 1
        for epoch in range(1, epochs + 1):
            self.net.train()
            self.train_accuracy.reset()
            total_loss = 0.0
            n = 0
            for features, label in train_loader:
                self.optimizer.zero_grad()  # Reset gradients
                output = self.net(features)
                # Calculate error and backpropagate
                loss = self.loss(output, label)
                output = torch.sigmoid(output)
                loss.backward()
                self.optimizer.step()  # Update weights with gradients

                self.train_accuracy(output, label)
                total_loss += loss.item()
                n += 1

                self.train_writer.add_scalar("CrossEntropyLoss", loss, step)

                step += 1

            train_loss = total_loss / n
            train_acc = self.train_accuracy.compute().item()

            self.train_writer.add_scalar("Accuracy", train_acc, step)


            self.net.eval()
            self.valid_accuracy.reset()
            total_loss = 0.0
            n = 0
            with torch.no_grad():
                for features, label in valid_loader:
                    output = self.net(features)
                    loss = self.loss(output, label)
                    output = torch.sigmoid(output)

                    self.valid_accuracy(output, label)
                    total_loss += loss.item()
                    n += 1

            valid_loss = total_loss / n
            valid_acc = self.valid_accuracy.compute().item()

            self.valid_writer.add_scalar("CrossEntropyLoss", valid_loss, step)
            self.valid_writer.add_scalar("Accuracy", valid_acc, step)

            print(f"{epoch=}, {train_loss=:.3f}, {valid_loss=:.3f}, {train_acc=:.3f}, {valid_acc=:.3f}")

In [ ]:
deep_fm = DeepFM(
    NUM_COLS,
    N_CATEGORIES_OF,
    EMB_SIZE,
    N_LAYERS,
    N_NEURONS,
    DEEP_OUTPUT_SIZE,
    N_CLASSES,
    LEARNING_RATE,
)
deep_fm.fit(train_dataloader, valid_dataloader, EPOCHS)

epoch=1, train_loss=0.652, valid_loss=0.500, train_acc=0.671, valid_acc=0.781
epoch=2, train_loss=0.450, valid_loss=0.407, train_acc=0.798, valid_acc=0.819
epoch=3, train_loss=0.396, valid_loss=0.377, train_acc=0.820, valid_acc=0.828
epoch=4, train_loss=0.373, valid_loss=0.362, train_acc=0.827, valid_acc=0.833
epoch=5, train_loss=0.359, valid_loss=0.351, train_acc=0.832, valid_acc=0.836
epoch=6, train_loss=0.350, valid_loss=0.344, train_acc=0.835, valid_acc=0.840
epoch=7, train_loss=0.342, valid_loss=0.338, train_acc=0.838, valid_acc=0.844
epoch=8, train_loss=0.337, valid_loss=0.335, train_acc=0.841, valid_acc=0.846
epoch=9, train_loss=0.333, valid_loss=0.332, train_acc=0.844, valid_acc=0.847
epoch=10, train_loss=0.330, valid_loss=0.330, train_acc=0.845, valid_acc=0.848
epoch=11, train_loss=0.327, valid_loss=0.328, train_acc=0.846, valid_acc=0.848
epoch=12, train_loss=0.326, valid_loss=0.328, train_acc=0.847, valid_acc=0.850
epoch=13, train_loss=0.324, valid_loss=0.327, train_acc=0.849

In [ ]:
# %tensorboard --logdir runs